# Statistical Modeling & Risk-Based Pricing

This notebook builds predictive models for insurance risk and pricing optimization.

We focus on:
- Claim Severity Prediction (Regression)
- Claim Occurrence Prediction (Classification)
- Premium Optimization Framework

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np

from src.modeling import * 

## 1. Load Dataset
We load the cleaned dataset prepared in previous tasks.

In [5]:
df = pd.read_csv("../data/MachineLearningRating_V3/cleaned_insurance_data.csv")

df.head()

C:\Users\sumex\AppData\Local\Temp\ipykernel_18324\1637338662.py:1: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/MachineLearningRating_V3/cleaned_insurance_data.csv")


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims,LossRatio,Margin
0,145249,12827,2015-03-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0.0,21.929825
1,145249,12827,2015-05-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0.0,21.929825
2,145249,12827,2015-07-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0,NaN,0.000000
3,145255,12827,2015-05-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0,0.0,512.848070
4,145255,12827,2015-07-01,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0,NaN,0.000000


## 2. Feature Engineering
We create time-based and behavioral features to improve model performance.

In [6]:
df = clean_data(df)
df = feature_engineering(df)
df.head()

,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims,LossRatio,Margin,TransactionYear,TransactionMonthNum,VehicleAge,ClaimOccurred
0,145249,12827,2015-03-01,True,NaN,Close Corporation,Mr,English,First National Bank,Current account,...,Commercial,IFRS Constant,21.929825,0.0,0.0,21.929825,2015,3,11,0
1,145249,12827,2015-05-01,True,NaN,Close Corporation,Mr,English,First National Bank,Current account,...,Commercial,IFRS Constant,21.929825,0.0,0.0,21.929825,2015,5,11,0
2,145249,12827,2015-07-01,True,NaN,Close Corporation,Mr,English,First National Bank,Current account,...,Commercial,IFRS Constant,0.000000,0.0,NaN,0.000000,2015,7,11,0
3,145255,12827,2015-05-01,True,NaN,Close Corporation,Mr,English,First National Bank,Current account,...,Commercial,IFRS Constant,512.848070,0.0,0.0,512.848070,2015,5,11,0
4,145255,12827,2015-07-01,True,NaN,Close Corporation,Mr,English,First National Bank,Current account,...,Commercial,IFRS Constant,0.000000,0.0,NaN,0.000000,2015,7,11,0


## Regression

In [7]:
X_train, X_test, y_train, y_test, preprocessor = prepare_data(
    df,
    target="TotalClaims",
    drop_cols=["ClaimOccurred"]
)

In [8]:
reg_results, reg_models = train_regression_models(
    X_train, X_test, y_train, y_test
)

reg_results

c:\Users\sumex\Desktop\insurance-risk-analytics\venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['NumberOfVehiclesInFleet']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\sumex\Desktop\insurance-risk-analytics\venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['NumberOfVehiclesInFleet']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\sumex\Desktop\insurance-risk-analytics\venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['NumberOfVehiclesInFleet']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


: 

In [ ]:
reg_results.sort_values("RMSE")

## Classification

In [ ]:
X_train, X_test, y_train, y_test = prepare_data(
    df,
    target="ClaimOccurred",
    drop_cols=["TotalClaims"]
)

clf_results, clf_models = train_classification_models(
    X_train, X_test, y_train, y_test
)

clf_results

## 3. Claim Severity Modeling
We predict TotalClaims for policies where claims occurred.

In [5]:
claims_df = df[df["TotalClaims"] > 0].copy()

## Feature Selection

In [6]:
features = [

    # Numeric
    "VehicleAge",
    "CustomValueEstimate",
    "Cylinders",
    "cubiccapacity",
    "kilowatts",
    "NumberOfDoors",

    # Categorical
    "Province",
    "VehicleType",
    "make",
    "bodytype",
    "CoverType"
]

In [7]:
numeric_features = [
    "VehicleAge",
    "CustomValueEstimate",
    "Cylinders",
    "cubiccapacity",
    "kilowatts",
    "NumberOfDoors"
]

In [8]:
categorical_features = [
    "Province",
    "VehicleType",
    "make",
    "bodytype",
    "CoverType"
]

## Train Test Split

In [9]:
X_train, X_test, y_train, y_test = prepare_data(
    claims_df,
    features=features,
    target="TotalClaims"
)

## Build Preprocessor

In [10]:
preprocessor = build_preprocessor(
    numeric_features,
    categorical_features
)

## Train Regression Models

In [11]:
reg_results, reg_models = train_regression_models(
    X_train,
    X_test,
    y_train,
    y_test,
    preprocessor
)

In [12]:
reg_results

,Model,RMSE,R2
0,Linear Regression,35283.020998,0.225933
1,Random Forest,37041.690823,0.146844
2,XGBoost,37645.439065,0.118806


## Classification Dataset

In [13]:
classification_features = features

In [14]:
X_train_cls, X_test_cls, y_train_cls, y_test_cls = prepare_data(
    df,
    features=classification_features,
    target="ClaimOccurred"
)

## Classification Models

In [15]:
cls_results, cls_models = train_classification_models(
    X_train_cls,
    X_test_cls,
    y_train_cls,
    y_test_cls,
    preprocessor
)

In [16]:
cls_results

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.997095,0.0,0.0,0.0
1,Random Forest,0.997090,0.0,0.0,0.0
2,XGBoost,0.997095,0.0,0.0,0.0


## SHAP Analysis

In [17]:
best_model = reg_models["XGBoost"]

In [ ]:
sample_data = X_test.sample(200, random_state=42)

: 

In [ ]:
shap_values = shap_analysis(
    best_model,
    sample_data
)

: 

# Premium Optimization Example

In [ ]:
probability_of_claim = 0.35

predicted_severity = 12000

In [ ]:
premium = calculate_optimized_premium(
    probability=probability_of_claim,
    severity=predicted_severity
)

premium

# Business Interpretation

## Key Findings

- Vehicle age strongly influences claim severity.
- Higher vehicle value increases expected claim cost.
- Certain vehicle types show consistently higher risk.
- SHAP analysis provides interpretable evidence for pricing decisions.

## Conclusion

The models demonstrate the feasibility of a dynamic risk-based pricing framework using policyholder and vehicle information.